# ATM Cash Demand Predictor — Data Processing & EDA

**Project:** AI-Powered ATM Cash Demand Forecasting and Replenishment Optimization

This notebook processes the supplied ATM CSV file step by step.

### Workflow
1. Import libraries
2. Load the CSV
3. Understand the dataset
4. Clean column names and data types
5. Process the `Date` column
6. Check missing values
7. Check duplicates
8. Validate ATM capacity, balance, and replenishment logic
9. Create useful date features
10. Perform basic EDA
11. Create an ATM-level summary
12. Save the cleaned dataset

> Run each cell from top to bottom. The comments explain what each step does.


## 1. Import required libraries

We use Pandas for data handling, NumPy for numerical operations, and Matplotlib for charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")


In [ ]:
import pandas as pd

## 2. Load the CSV file

The notebook reads the uploaded ATM dataset. Change `file_path` only if you move the CSV to another folder.

In [ ]:
file_path = r"/Users/ranger/Desktop/HCL.Project/atm_cash_management_dataset_final (6) (1).csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


## 3. View the data

`head()` shows the first records. This is the first visual check that the CSV was loaded correctly.

In [ ]:
df.head()


## 4. Check the size of the dataset

In [ ]:
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)


## 5. Check column names

This helps us understand what information is available.

In [ ]:
print(df.columns.tolist())


## 6. Check data types and non-null values

`info()` tells us whether columns are numeric, text, or another type and whether values are missing.

In [ ]:
df.info()


## 7. Basic statistical summary

This gives count, mean, standard deviation, minimum, maximum, and percentiles for numerical columns.

In [ ]:
df.describe()


## 8. Process the Date column

The `Date` column is converted from text into Pandas datetime format. Invalid dates become `NaT`, which can then be checked.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Date data type:", df["Date"].dtype)
print("Invalid/missing dates:", df["Date"].isna().sum())


## 9. Extract useful date features

Date information is very useful for ATM demand analysis. We create year, month, day, quarter, week, weekday number, and weekend indicators.

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Quarter"] = df["Date"].dt.quarter
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype("Int64")
df["Day_of_Week_Number"] = df["Date"].dt.dayofweek
df["Is_Weekend"] = (df["Day_of_Week_Number"] >= 5).astype(int)

df[[
    "Date", "Year", "Month", "Day", "Quarter",
    "Week_of_Year", "Day_of_Week_Number", "Is_Weekend"
]].head()


## 10. Check the date range

This tells us the earliest and latest date available in the dataset.

In [ ]:
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())


## 11. Check missing values

Missing values can cause problems during analysis and machine learning.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)

print("Missing values by column:")
print(missing)

print("\nTotal missing values:", missing.sum())


## 12. Handle missing values

For numeric columns we use the median. For categorical columns we use the most frequent value (mode). Date values are not automatically filled because an incorrect date could create misleading time features.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="object").columns

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after treatment:")
print(df.isnull().sum().sort_values(ascending=False).head())


## 13. Check duplicate rows

Duplicate records can distort averages and model training.

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)


## 14. Remove exact duplicate rows

Only exact duplicate records are removed here.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after duplicate removal:", df.shape)


## 15. Check categorical values

This helps detect spelling inconsistencies or unexpected categories.

In [ ]:
for col in [
    "Day_of_Week",
    "Time_of_Day",
    "Location_Type",
    "Weather_Condition",
    "Replenishment_Required"
]:
    print("\n", col)
    print(df[col].value_counts(dropna=False))


## 16. Validate ATM capacity

Each `ATM_ID` should have one fixed capacity. We check how many different capacities each ATM has.

In [ ]:
capacity_check = df.groupby("ATM_ID")["ATM_Capacity"].nunique()

print("ATM IDs with more than one capacity:")
print(capacity_check[capacity_check > 1])

print("\nNumber of unique ATM IDs:", df["ATM_ID"].nunique())
print("Number of unique ATM capacities:", df[["ATM_ID", "ATM_Capacity"]].drop_duplicates()["ATM_Capacity"].nunique())


## 17. Validate ATM balance

Business rule used in this project:

**ATM Balance = ATM Capacity − Total Withdrawals**

If withdrawal is greater than capacity, available balance is treated as zero.

In [ ]:
expected_balance = (
    df["ATM_Capacity"] - df["Total_Withdrawals"]
).clip(lower=0)

balance_errors = (df["ATM_Balance"] != expected_balance).sum()

print("Balance calculation errors:", balance_errors)


## 18. Validate replenishment logic

Replenishment should occur only when withdrawal demand is greater than ATM capacity.

In [ ]:
expected_replenishment = df["Total_Withdrawals"] > df["ATM_Capacity"]
actual_replenishment = df["Replenishment_Amount"] > 0

logic_errors = (expected_replenishment != actual_replenishment).sum()

print("Replenishment logic errors:", logic_errors)
print("Records requiring replenishment:", actual_replenishment.sum())


## 19. Check impossible numeric values

We check for negative withdrawals, deposits, capacity, and balance.

In [ ]:
checks = {
    "Negative withdrawals": (df["Total_Withdrawals"] < 0).sum(),
    "Negative deposits": (df["Total_Deposits"] < 0).sum(),
    "Negative capacity": (df["ATM_Capacity"] < 0).sum(),
    "Negative balance": (df["ATM_Balance"] < 0).sum(),
    "Negative replenishment": (df["Replenishment_Amount"] < 0).sum(),
}

pd.Series(checks, name="Count")


## 20. Analyze withdrawal distribution

A histogram helps us understand typical withdrawal values and possible extreme observations.

In [ ]:
plt.hist(df["Total_Withdrawals"], bins=30)
plt.title("Distribution of ATM Withdrawals")
plt.xlabel("Total Withdrawals")
plt.ylabel("Frequency")
plt.show()


## 21. Analyze next-day cash demand

`Cash_Demand_Next_Day` is a useful target for the future forecasting model.

In [ ]:
plt.hist(df["Cash_Demand_Next_Day"], bins=30)
plt.title("Distribution of Next-Day Cash Demand")
plt.xlabel("Cash Demand Next Day")
plt.ylabel("Frequency")
plt.show()


## 22. Average cash demand by day of week

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_demand = (
    df.groupby("Day_of_Week")["Cash_Demand_Next_Day"]
    .mean()
    .reindex(day_order)
)

print(day_demand)

day_demand.plot(kind="bar")
plt.title("Average Next-Day Cash Demand by Day")
plt.xlabel("Day of Week")
plt.ylabel("Average Cash Demand")
plt.xticks(rotation=45)
plt.show()


## 23. Weekend vs weekday analysis

In [ ]:
weekend_demand = (
    df.groupby("Is_Weekend")["Cash_Demand_Next_Day"]
    .mean()
)

print(weekend_demand)

weekend_demand.plot(kind="bar")
plt.title("Average Cash Demand: Weekday vs Weekend")
plt.xlabel("Weekend Flag")
plt.ylabel("Average Cash Demand")
plt.xticks([0, 1], ["Weekday", "Weekend"], rotation=0)
plt.show()


## 24. Monthly demand analysis

In [ ]:
monthly_demand = (
    df.groupby("Month")["Cash_Demand_Next_Day"]
    .mean()
)

print(monthly_demand)

monthly_demand.plot(marker="o")
plt.title("Average Next-Day Cash Demand by Month")
plt.xlabel("Month")
plt.ylabel("Average Cash Demand")
plt.xticks(range(1, 13))
plt.grid(True)
plt.show()


## 25. ATM-wise demand analysis

In [ ]:
atm_demand = (
    df.groupby("ATM_ID")["Cash_Demand_Next_Day"]
    .mean()
    .sort_values(ascending=False)
)

print("Top 10 ATM IDs by average next-day demand:")
print(atm_demand.head(10))

atm_demand.plot(kind="bar", figsize=(14, 6))
plt.title("Average Next-Day Cash Demand by ATM")
plt.xlabel("ATM ID")
plt.ylabel("Average Cash Demand")
plt.xticks(rotation=90)
plt.show()


## 26. Replenishment analysis

This identifies how often ATMs need additional cash and the total amount required.

In [ ]:
replenishment_cases = (df["Replenishment_Amount"] > 0).sum()
total_replenishment = df["Replenishment_Amount"].sum()
replenishment_percentage = replenishment_cases / len(df) * 100

print("Replenishment cases:", replenishment_cases)
print("Replenishment percentage:", round(replenishment_percentage, 2), "%")
print("Total replenishment amount:", total_replenishment)


## 27. Location-wise demand analysis

In [ ]:
location_demand = (
    df.groupby("Location_Type")["Cash_Demand_Next_Day"]
    .mean()
    .sort_values(ascending=False)
)

print(location_demand)

location_demand.plot(kind="bar")
plt.title("Average Next-Day Cash Demand by Location Type")
plt.xlabel("Location Type")
plt.ylabel("Average Cash Demand")
plt.xticks(rotation=45)
plt.show()


## 28. Correlation analysis

Correlation helps us understand numerical relationships. A value near +1 indicates a strong positive relationship, near -1 a strong negative relationship, and near 0 a weak linear relationship.

In [ ]:
numeric_features = [
    "Total_Withdrawals",
    "Total_Deposits",
    "Previous_Day_Cash_Level",
    "Nearby_Competitor_ATMs",
    "Cash_Demand_Next_Day",
    "ATM_Capacity",
    "ATM_Balance",
    "Replenishment_Amount"
]

correlation = df[numeric_features].corr()

display(correlation.round(2))

plt.figure(figsize=(11, 8))
plt.imshow(correlation, aspect="auto")
plt.colorbar()

plt.xticks(
    range(len(correlation.columns)),
    correlation.columns,
    rotation=90
)
plt.yticks(
    range(len(correlation.columns)),
    correlation.columns
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 29. Create an ATM-level business summary

This table is useful for management reporting and identifying ATMs that need more cash planning.

In [ ]:
atm_summary = df.groupby("ATM_ID").agg(
    Average_Withdrawal=("Total_Withdrawals", "mean"),
    Average_Next_Day_Demand=("Cash_Demand_Next_Day", "mean"),
    ATM_Capacity=("ATM_Capacity", "first"),
    Average_ATM_Balance=("ATM_Balance", "mean"),
    Total_Replenishment=("Replenishment_Amount", "sum"),
    Replenishment_Cases=("Replenishment_Amount", lambda x: (x > 0).sum())
).sort_values("Total_Replenishment", ascending=False)

atm_summary.head(10)


## 30. Final data-quality check

In [ ]:
print("Final dataset shape:", df.shape)

print("\nTotal missing values:", df.isnull().sum().sum())
print("Total duplicate rows:", df.duplicated().sum())

print("\nFinal columns:")
print(df.columns.tolist())


## 31. Save the cleaned dataset

The processed dataset is saved as a new CSV so the original file remains unchanged.

In [ ]:
output_file = r"/mnt/data/ATM_Cash_Demand_Cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved to:")
print(output_file)


# Final Outcome

After completing this notebook, you have:

- Converted `Date` into a proper datetime format
- Created useful date features
- Checked and handled missing values
- Removed duplicate rows
- Checked categorical values
- Validated ATM capacity
- Validated ATM balance calculations
- Validated replenishment logic
- Checked invalid numerical values
- Performed basic EDA
- Analyzed demand by day, month, weekend, ATM, and location
- Analyzed replenishment requirements
- Created an ATM-level summary
- Saved a clean CSV for further machine-learning work

### Next step

The cleaned dataset can now be used for **feature engineering → train/test split → ATM cash-demand prediction → model evaluation → replenishment optimization**.
